# Dataset download
Dataset can be downloaded from the code below.
However we should downloadn the labels manually.  

In [ ]:
import utils.dataset
utils.dataset.download_dataset(output_dir="./data")

In [2]:
import pathlib
import yaml
import utils.dataset
import IPython.display

with open("data/cxr_dataset.yaml", "r") as f:
    cfg = yaml.safe_load(f)

image_root = pathlib.Path(cfg['image_root'])

# 3. Load dataset splits
train_df, train_paths, train_labels = utils.dataset.load_split(cfg['train_csv'], image_root)
valid_df, valid_paths, valid_labels = utils.dataset.load_split(cfg['valid_csv'], image_root)
test_df,  test_paths,  test_labels  = utils.dataset.load_split(cfg['test_csv'],  image_root)

# IPython.display.display(train_df)
# IPython.display.display(train_paths)
# IPython.display.display(train_labels)

# print("Train:", len(train_df))
# print("Valid:", len(valid_df))
# print("Test :", len(test_df))

# 5. Apply filtering
train_df_filtered, train_paths_filtered = utils.dataset.filter_dataset(train_df, train_paths, cfg['top_labels'], cfg['all_labels'])
valid_df_filtered, valid_paths_filtered = utils.dataset.filter_dataset(valid_df, valid_paths, cfg['top_labels'], cfg['all_labels'])
test_df_filtered,  test_paths_filtered  = utils.dataset.filter_dataset(test_df,  test_paths,  cfg['top_labels'], cfg['all_labels'])
labels                                  = train_df_filtered[cfg['top_labels']].values.astype('float32')
# IPython.display.display(train_df_filtered)
# IPython.display.display(train_paths_filtered)

IPython.display.display(labels[:5])
print("Train:", len(train_df_filtered))
print("Valid:", len(valid_df_filtered))
print("Test :", len(test_df_filtered))


array([[1., 0., 0.],
       [1., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], dtype=float32)

Train: 12980
Valid: 1768
Test : 4025


In [3]:
import utils.models
import utils.utils

# 1. Lock random seeds for reproducibility
utils.utils.set_random_seeds(seed=42)

# 2. Choose your model from Hugging Face
model_name = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
# model_name = 'hf-hub:luhuitong/CLIP-ViT-L-14-448px-MedICaT-ROCO'

model, preprocess, tokenizer, device = utils.models.load_clip_model(model_name=model_name,
                                                                    freeze_backbone=True)

# 3. Package the filtered datasets into clean dictionaries
paths_dict = {'train': train_paths_filtered,
              'valid': valid_paths_filtered,
              'test' : test_paths_filtered}

df_dict = {'train': train_df_filtered, 
           'valid': valid_df_filtered,
           'test' : test_df_filtered}

# 4. Create PyTorch DataLoaders using your custom module
train_loader, valid_loader, test_loader = utils.dataset.create_dataloaders(
    paths_dict=paths_dict, 
    df_dict=df_dict, 
    top_labels=cfg['top_labels'], 
    preprocess=preprocess, 
    batch_size=16,
    num_workers=2
)

Random seeds locked to 42.
Loading hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224 onto cuda...
Model backbone parameters frozen (requires_grad = False).


AttributeError: module 'utils.dataset' has no attribute 'create_dataloaders'

### Loading open_clip pre-trained model

In [ ]:
import open_clip
import torch
from urllib.request import urlopen
from PIL import Image

model, _ , preprocess = open_clip.create_model_and_transforms('hf-hub:luhuitong/CLIP-ViT-L-14-448px-MedICaT-ROCO')
tokenizer             = open_clip.get_tokenizer('hf-hub:luhuitong/CLIP-ViT-L-14-448px-MedICaT-ROCO')

model, processor = open_clip.create_model_from_pretrained('hf-hub:xcwangpsu/MedCSP_clip')
tokenizer_text   = open_clip.get_tokenizer('hf-hub:xcwangpsu/MedCSP_clip')




import requests
from PIL import Image
import matplotlib.pyplot as plt

from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("flaviagiammarino/pubmed-clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("flaviagiammarino/pubmed-clip-vit-base-patch32")




from PIL import Image
import requests

from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")




import torch
from urllib.request import urlopen
from PIL import Image
from open_clip import create_model_from_pretrained, get_tokenizer

# Load the model and config files from the Hugging Face Hub
model, preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



for p in model.parameters():
    p.requires_grad = False
    
    
    
    
import torch
import random
import numpy as np
from torch.utils.data import DataLoader

# Fix random seed
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# For deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# Create generator for DataLoader shuffle
g = torch.Generator()
g.manual_seed(seed)


train_dataset = XrayDataset(
    train_paths_filtered,
    train_df_filtered,
    top_labels,
    preprocess
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    generator=g
)


valid_dataset = XrayDataset(
    valid_paths_filtered,
    valid_df_filtered,
    top_labels,
    preprocess
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    generator=g
)


test_dataset = XrayDataset(
    test_paths_filtered,
    test_df_filtered,
    top_labels,
    preprocess
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    generator=g
)